# Session 3 · Part 3 — Interpret inferred molecular landscapes

**Goal:** move from scores to biological and technical interpretation. We select a well-predicted protein,
identify the RNA feature most associated with its measured abundance, compare four spatial maps, and
derive an exploratory embedding from the complete inferred protein panel.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Select a protein and a transcript using explicit evidence


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_table
from dgat_tutorial.evaluation import protein_correlations
from dgat_tutorial.plotting import plot_spatial_feature
from dgat_tutorial.processing import normalize_total_log1p

dataset = load_tutorial_data(paths.raw_data)
predicted = load_prediction_table(str(preferred_prediction_path(paths)))
common_spots = dataset.spots.index.intersection(dataset.proteins.index).intersection(predicted.index)
if common_spots.empty:
    raise ValueError("Observed and predicted data have no shared spot IDs.")
spots = dataset.spots.loc[common_spots]
observed = dataset.proteins.loc[common_spots]
predicted = predicted.loc[common_spots]
transcripts = normalize_total_log1p(dataset.transcripts.loc[common_spots].select_dtypes(include=[np.number]))

correlations = protein_correlations(observed, predicted)
protein = correlations.iloc[0]["protein"]
gene_correlations = transcripts.corrwith(observed[protein]).dropna().sort_values(key=abs, ascending=False)
gene = gene_correlations.index[0]
print(f"Selected protein {protein} (best Pearson); associated transcript {gene} (|r| maximum).")


### Figure 14 — Transcript, measured protein, inferred protein, and residual


In [ ]:
residual = predicted[protein] - observed[protein]
limit = float(np.abs(residual).max())
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
plot_spatial_feature(spots, transcripts[gene], f"Normalized RNA: {gene}", cmap="magma", ax=axes[0])
plot_spatial_feature(spots, observed[protein], f"Observed: {protein}", ax=axes[1])
plot_spatial_feature(spots, predicted[protein], f"Predicted: {protein}", ax=axes[2])
residual_scatter = axes[3].scatter(spots["x"], spots["y"], c=residual, cmap="coolwarm", vmin=-limit, vmax=limit, s=24)
axes[3].set(title="Residual (predicted − observed)", xlabel="x", ylabel="y", aspect="equal")
plt.colorbar(residual_scatter, ax=axes[3], fraction=0.046, pad=0.04)
landscape_path = paths.figures / "session03_landscape_comparison.png"
fig.tight_layout(); fig.savefig(landscape_path, dpi=160, bbox_inches="tight"); plt.show()


**How to read it:** spatially localized residuals may reveal tissue boundaries, composition shifts,
antibody effects, or a domain shift. Transcript–protein discordance can be biological because translation,
trafficking, and degradation separate RNA abundance from surface-protein abundance.


### Figure 15 — Downstream structure inferred from the full protein panel


In [ ]:
scaled = (predicted - predicted.mean(axis=0)) / (predicted.std(axis=0) + 1e-8)
embedding = PCA(n_components=2, random_state=7).fit_transform(scaled)
n_clusters = min(6, max(2, len(predicted) // 100))
clusters = KMeans(n_clusters=n_clusters, n_init=20, random_state=7).fit_predict(embedding)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(embedding[:, 0], embedding[:, 1], c=clusters, cmap="tab10", s=14)
axes[0].set(title="PCA of inferred protein panel", xlabel="PC1", ylabel="PC2")
axes[1].scatter(spots["x"], spots["y"], c=clusters, cmap="tab10", s=18)
axes[1].set(title="Inferred-protein clusters in tissue", xlabel="x", ylabel="y", aspect="equal")
embedding_path = paths.figures / "session03_inferred_protein_embedding.png"
fig.tight_layout(); fig.savefig(embedding_path, dpi=160, bbox_inches="tight"); plt.show()


These clusters are hypotheses, not validated cell types. Annotate them only after checking marker panels,
spatial context, robustness to cluster count, and agreement with observed modalities or orthogonal data.


In [ ]:
prompts = (
    "# Session 3 interpretation prompts\n\n"
    "- Which proteins are accurate pointwise but spatially over-smoothed?\n"
    "- Where are residuals spatially localized, and what technical or biological process could explain them?\n"
    "- Which multi-protein clusters are stable and supported by known marker combinations?\n"
    "- What donor, batch, antibody, tissue-boundary, or cell-composition effects could mislead evaluation?\n"
    "- What held-out dataset would best test generalization?\n"
)
prompt_path = paths.results / "session03_interpretation_prompts.md"
prompt_path.write_text(prompts, encoding="utf-8")
manifest = write_checkpoint(
    "3.3", [landscape_path, embedding_path, prompt_path],
    summary={"transcript": gene, "protein": protein, "exploratory_clusters": n_clusters}, start=paths.root,
)
print(prompts); print(f"Checkpoint written: {manifest}")


## Next steps

A complete analysis now has three evidence layers: per-protein accuracy, spatial fidelity, and biological
interpretation. Before publication, repeat all three on held-out biological samples and include uncertainty
or replicate variability—not only a single fitted map.
